# Notebook 03 — Model Training

Goals:
- Inspect the Small3DUNet architecture
- Run a short training sanity-check (2 epochs, 4 samples)
- Load the best saved checkpoint and evaluate on the validation set
- Visualise training curves and sample predictions

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader

from src.data.loader import build_index, train_val_split
from src.data.dataset import BrainCTDataset
from src.models.unet import Small3DUNet
from src.training.trainer import train, evaluate, dice_score
from src.visualization.plots import show_training_curves, show_prediction

DATA_ROOT  = '../data/raw'
SAVE_PATH  = '../models/best_model.pth'

device = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'Device: {device}')

ModuleNotFoundError: No module named 'torch'

## 1. Model architecture

In [ ]:
model = Small3DUNet()
print(model)
print(f'\nTotal parameters: {model.count_parameters():,}')

# Forward pass sanity check
dummy = torch.zeros(1, 1, 64, 128, 128)
out   = model(dummy)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {out.shape}  (should match input spatial dims)')
print(f'Output range : [{out.min():.4f}, {out.max():.4f}]  (sigmoid → [0, 1])')

## 2. Sanity-check training (2 epochs, tiny subset)

Before running a full training job, verify that:
- Loss decreases at least a little in 2 epochs
- No NaN / inf in loss
- Dice score is computed without errors

In [ ]:
records = build_index(DATA_ROOT)
train_records, val_records = train_val_split(records, val_fraction=0.2)

# Use 4 train + 2 val for quick sanity check
tiny_train = DataLoader(BrainCTDataset(train_records[:4], augment=False), batch_size=2)
tiny_val   = DataLoader(BrainCTDataset(val_records[:2],   augment=False), batch_size=2)

sanity_model = Small3DUNet().to(device)
history = train(
    model=sanity_model,
    train_loader=tiny_train,
    val_loader=tiny_val,
    epochs=2,
    lr=1e-3,
    save_path='/tmp/sanity_model.pth',
    device=device,
)
print('Sanity check passed ✓')

## 3. Full training

Run from the terminal for a clean background process:
```bash
python scripts/train.py \
  --data-root data/raw \
  --save-path models/best_model.pth \
  --epochs 50 --batch-size 2 --lr 1e-3
```

Expected time: ~2–4h on CPU | ~40–60min on Apple M-series (MPS)

## 4. Load checkpoint and evaluate

In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=device)
model = Small3DUNet().to(device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Checkpoint from epoch : {checkpoint['epoch']}")
print(f"Saved val Dice        : {checkpoint['val_dice']:.4f}")

In [ ]:
records = build_index(DATA_ROOT)
_, val_records = train_val_split(records, val_fraction=0.2)
val_loader = DataLoader(BrainCTDataset(val_records, augment=False), batch_size=2)

val_loss, val_dice = evaluate(model, val_loader, device)
print(f'Validation loss : {val_loss:.4f}')
print(f'Validation Dice : {val_dice:.4f}')

## 5. Sample predictions

In [ ]:
import numpy as np
from src.data.loader import load_nifti
from src.preprocessing.transforms import preprocess

model.eval()
for i, r in enumerate(val_records[:3]):
    vol_raw, spacing = load_nifti(r['image'])
    msk_raw, _       = load_nifti(r['mask'])

    vol_pp, msk_pp = preprocess(vol_raw, msk_raw)
    x = torch.from_numpy(vol_pp).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x)[0, 0].cpu().numpy()

    pred_bin = (pred >= 0.5).astype(np.uint8)
    print(f'Sample {i+1}: Dice = {dice_score(torch.from_numpy(pred), torch.from_numpy(msk_pp.astype(np.float32))):.3f}')
    show_prediction(vol_pp, pred_bin, true_mask=msk_pp)